## Cross-Category Embedding Scatterplot

A single 2D scatterplot showing all 10 categories together, colored by category, so you can see which categories sit close to each other in embedding space (topically related) versus far apart (topically distinct). Run separately for abstracts and PDF chunks.

**This is a different UMAP fit from the BERTopic notebook, on purpose**: the BERTopic notebook fits UMAP once per category, which is correct for finding within-category sub-topics but means each category's 2D coordinates live in their own separate space, not comparable across categories. This notebook fits UMAP once globally across all 10 categories combined, so every point's position is relative to every other point regardless of category. This is the only place in the pipeline where categories are intentionally combined into one embedding space, everywhere else (clustering, QA generation, database benchmarking) stays strictly per-category. Worth stating this explicitly if the figure goes into the paper, so a reader doesn't assume it reflects the same per-category clustering methodology used elsewhere.

**Why UMAP and not PCA/t-SNE for this**: UMAP is already the dimensionality-reduction tool used elsewhere in this project (BERTopic), so reusing it keeps the methodology consistent rather than introducing a third reduction technique. It also tends to preserve both local neighborhood structure and a reasonable sense of global relative distance between clusters, which is exactly what this plot needs to show, unlike t-SNE, which is known to distort inter-cluster distances (two clusters being close or far apart in a t-SNE plot isn't reliably meaningful), UMAP's global structure is comparatively more trustworthy for this purpose, though still an approximation, not literal distance.

**No sampling**: every available point is plotted (one per paper for abstracts, one per page for PDF chunks). At tens of thousands of points, matplotlib's `scatter` with small point size and moderate alpha handles this fine and stays honest about density, rather than a sampled plot that could visually understate how much a category actually overlaps with another.

### Install dependencies (Colab only, skip if already installed locally)

In [1]:
# Uncomment if running in Colab or a fresh environment
# !pip install umap-learn huggingface_hub matplotlib numpy -q

### Configuration

In [2]:
import os

# --- Hugging Face repo (shared corpus) ---
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"
REPO_TYPE = "dataset"

# --- Which model's embeddings to use ---
EMBEDDING_MODEL = "text-embedding-3-small"

# --- The 10 target categories ---
ALL_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

# Distinct, colorblind-safe-ish palette, one fixed color per category so abstracts
# and PDF chunk plots use the same color for the same category (easier to compare the two figures)
CATEGORY_COLORS = {
    "cs.AI": "#4C72B0", "cs.LG": "#DD8452", "cs.IR": "#55A868", "cs.DB": "#C44E52", "cs.SE": "#8172B2",
    "cs.CV": "#937860", "cs.CL": "#DA8BC3", "cs.NE": "#8C8C8C", "cs.DC": "#CCB974", "cs.CR": "#64B5CD",
}

# --- UMAP settings, global fit across all categories ---
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1   # slightly looser than the BERTopic clustering UMAP (min_dist=0.0), since this is
                       # for visual readability, not feeding a downstream clustering step
RANDOM_STATE = 42

# --- Visualization settings, sized for IEEE two-column format ---
IEEE_COLUMN_WIDTH_INCHES = 7.16   # full double-column width, this plot needs the room given 10 categories
IEEE_FIGURE_DPI = 300
POINT_SIZE = 3
POINT_ALPHA = 0.35

# --- Local paths ---
PROJECT_ROOT = ".."
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

### Authenticate with Hugging Face

In [3]:
from huggingface_hub import whoami, login

try:
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticated as: Sakhiur


### Loader: pull embeddings for every available category

Returns vectors and a parallel array of category labels. Missing categories (your collaborator's, until uploaded) are skipped and reported, the plot simply shows fewer categories rather than failing.

In [4]:
import json
import numpy as np
from huggingface_hub import hf_hub_download


def load_all_categories(corpus_type: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns (vectors, labels) stacked across every available category.
    vectors: (N, dim) float32 array
    labels: (N,) array of category strings, parallel to vectors
    """
    all_vectors = []
    all_labels = []

    for category in ALL_CATEGORIES:
        repo_path = f"embeddings/{EMBEDDING_MODEL}/{corpus_type}/{category}.jsonl"
        try:
            local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
        except Exception:
            print(f"  [SKIP] {category}: not found on Hub")
            continue

        count = 0
        with open(local_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                all_vectors.append(record["embedding"])
                all_labels.append(category)
                count += 1

        print(f"  {category}: {count} points")

    vectors = np.array(all_vectors, dtype=np.float32)
    labels = np.array(all_labels)
    return vectors, labels

### Plot function

One UMAP fit across the combined vectors, one scatter call per category (so each gets its own color and legend entry, drawn in a fixed category order for a reproducible layering, later categories in the list are drawn on top of earlier ones, which matters when points overlap heavily).

In [5]:
import matplotlib.pyplot as plt
from umap import UMAP

plt.rcParams.update({
    "font.size": 9,
    "font.family": "serif",
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})


def plot_cross_category_scatter(vectors: np.ndarray, labels: np.ndarray, corpus_type: str, out_path: str) -> np.ndarray:
    print(f"  Fitting UMAP on {vectors.shape[0]} points, dim={vectors.shape[1]} -> 2D...")
    reducer = UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        n_components=2,
        min_dist=UMAP_MIN_DIST,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    coords_2d = reducer.fit_transform(vectors)

    fig, ax = plt.subplots(figsize=(IEEE_COLUMN_WIDTH_INCHES, IEEE_COLUMN_WIDTH_INCHES * 0.75))

    for category in ALL_CATEGORIES:
        mask = labels == category
        if not mask.any():
            continue
        ax.scatter(
            coords_2d[mask, 0], coords_2d[mask, 1],
            c=CATEGORY_COLORS[category], s=POINT_SIZE, alpha=POINT_ALPHA,
            label=f"{category} (n={mask.sum()})", linewidths=0,
        )

    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_title(f"Cross-category embedding space ({corpus_type}, {EMBEDDING_MODEL})")
    ax.set_xticks([])
    ax.set_yticks([])
    legend = ax.legend(markerscale=4, loc="center left", bbox_to_anchor=(1.01, 0.5), framealpha=0.9)
    for handle in legend.legend_handles:
        handle.set_alpha(1.0)

    fig.tight_layout()
    fig.savefig(out_path, dpi=IEEE_FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

    return coords_2d

### Generate: abstracts

In [6]:
print("Loading abstract embeddings across all categories...")
abstract_vectors, abstract_labels = load_all_categories("abstracts")

print(f"\nTotal: {len(abstract_vectors)} points across {len(set(abstract_labels))} categories")

abstract_out_path = os.path.join(FIGURES_DIR, f"cross_category_scatter_abstracts_{EMBEDDING_MODEL}.png")
abstract_coords = plot_cross_category_scatter(abstract_vectors, abstract_labels, "abstracts", abstract_out_path)
print(f"Saved {abstract_out_path}")

Loading abstract embeddings across all categories...
  cs.AI: 7369 points


c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sakhiur\.cache\huggingface\hub\datasets--Sakhiur--empirical-rag-paradigm-benchmark. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  cs.LG: 10746 points
  cs.IR: 3076 points
  cs.DB: 1532 points
  cs.SE: 2281 points
  cs.CV: 14163 points
  cs.CL: 12418 points
  cs.NE: 1501 points
  cs.DC: 1517 points
  cs.CR: 1523 points

Total: 56126 points across 10 categories
  Fitting UMAP on 56126 points, dim=1536 -> 2D...


c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved ..\results\figures\cross_category_scatter_abstracts_text-embedding-3-small.png


### Generate: PDF chunks

In [7]:
print("Loading PDF chunk embeddings across all categories...")
pdf_vectors, pdf_labels = load_all_categories("pdf_chunks")

print(f"\nTotal: {len(pdf_vectors)} points across {len(set(pdf_labels))} categories")

pdf_out_path = os.path.join(FIGURES_DIR, f"cross_category_scatter_pdf_chunks_{EMBEDDING_MODEL}.png")
pdf_coords = plot_cross_category_scatter(pdf_vectors, pdf_labels, "pdf_chunks", pdf_out_path)
print(f"Saved {pdf_out_path}")

Loading PDF chunk embeddings across all categories...
  cs.AI: 2060 points
  cs.LG: 1937 points
  cs.IR: 1371 points
  cs.DB: 1768 points
  cs.SE: 1956 points
  cs.CV: 1250 points
  cs.CL: 1363 points
  cs.NE: 1668 points
  cs.DC: 1826 points
  cs.CR: 1517 points

Total: 16716 points across 10 categories
  Fitting UMAP on 16716 points, dim=1536 -> 2D...


c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved ..\results\figures\cross_category_scatter_pdf_chunks_text-embedding-3-small.png


### Summary

Point counts per category, per corpus, so you can see at a glance which categories are represented and whether the plot's density differences are just reflecting category size imbalance (e.g. if cs.LG has far more points than cs.DB, cs.LG will visually dominate regardless of topical spread, worth keeping in mind when reading the figure, dense-looking areas can mean either "topically concentrated" or "just more papers," this table helps you tell which).

In [8]:
print(f"{'category':<8} {'abstracts':<11} {'pdf_chunks'}")
print("-" * 32)
for category in ALL_CATEGORIES:
    n_abs = int(np.sum(abstract_labels == category)) if len(abstract_labels) else 0
    n_pdf = int(np.sum(pdf_labels == category)) if len(pdf_labels) else 0
    print(f"{category:<8} {n_abs:<11} {n_pdf}")

print(f"\nFigures saved to: {FIGURES_DIR}")

category abstracts   pdf_chunks
--------------------------------
cs.AI    7369        2060
cs.LG    10746       1937
cs.IR    3076        1371
cs.DB    1532        1768
cs.SE    2281        1956
cs.CV    14163       1250
cs.CL    12418       1363
cs.NE    1501        1668
cs.DC    1517        1826
cs.CR    1523        1517

Figures saved to: ..\results\figures
